# RTL track · Radeon Cloud 上手

本赛道要求做一个**在本地运行的智能体**：读一道硬件设计题，调用本地部署的开源大模型
写出 Verilog / SystemVerilog 代码，并借助 EDA 工具自行检查、修正。赛事方用同一套判定器给代码定级，
再与「同一模型不经智能体直接生成」的基线比较。

这个 notebook 带你在 Radeon Cloud 实例上把整条链路从头走一遍，每一步都说明：
**在做什么、用到仓库里哪个文件、输出是什么、怎么读这个输出。**

### 整体流程

```
  题目 tasks/<题>/prompt.txt
        │
        ▼
  example/run.sh ──► 智能体 agent/ ──► 本地模型（vLLM，由 serve/start.sh 启动）
        │               │  ▲
        │               ▼  │  报错信息交回模型，修正后重试
        │            EDA 工具检查（编译、综合）
        ▼
  产物 solution.v + trace.jsonl（每一次模型与工具调用的记录）
        │
        ▼
  判定器 selftest/judge.py ──► 级别 L0–L3 ──► 系数
        │
        ▼
  计分 selftest/score.py ──► 能力分、增益分
```

基线走另一条更短的路：`example/run_baseline.sh` 把题面直接交给同一个模型，只生成一次，
不调用工具。方案比基线多出来的部分，就是你的智能体与技能包带来的提升。

### 步骤一览

| 步骤 | 内容 | 需要模型 | 预计耗时 |
| --- | --- | --- | --- |
| 0 | 配置 | 否 | 即时 |
| 1 | 检查实例环境 | 否 | 1 分钟 |
| 2 | 获取参考仓库 `rtlagent2026` | 否 | 1 分钟 |
| 3 | 仓库导览：每个目录、每个文件是做什么的 | 否 | 即时 |
| 4 | 安装示范方案的依赖 | 否 | 1 分钟 |
| 5 | 认识一道题：智能体能看到什么，看不到什么 | 否 | 即时 |
| 6 | 判定链路校验：用参考答案跑通判定器 | 否 | 约 3 分钟 |
| 7 | 桩模式：不用真模型，验证 `run.sh` 的输入输出 | 否 | 1 分钟 |
| 8 | 投题服务：评测当天赛事方怎么把题发给你 | 否 | 2 分钟 |
| 9 | 启动本地模型，跑一道题的方案与基线 | **是** | 10–20 分钟（首次含下载） |
| 10 | 完整自测与算分（可选） | **是** | 约 20 分钟 |
| 11 | 清理 | 否 | 即时 |

步骤 1–8 不需要模型，应当全部通过。**第 6 步不通过时，后面的判定结果都不可信**，请先
解决环境问题。

### 几个名词

| 名词 | 含义 |
| --- | --- |
| 方案 | 你的智能体加技能包，入口是 `example/run.sh` |
| 基线 | 赛事方提供的 `example/baseline.py`：同一模型、只看题面、单次生成、不调用工具。队伍不得修改 |
| 判定 | 用 EDA 工具给一份代码定级，L0 最低，L3 最高 |
| 题集得分 | 每道题按级别折算成 0–1 的系数，再对所有题取平均 |
| 能力分 | 30 × 方案的题集得分 |
| 增益分 | 归一化增益 g = (方案 − 基线) / (1 − 基线)，即方案补上了基线与满分之间差距的比例；增益分 = 40 × min(g / 0.6, 1) |

### 怎么运行

自上而下逐格执行（Shift + Enter）。以 `%%bash` 开头的单元格在独立的 shell 中运行，
彼此之间通过第 0 步写入的环境变量传递路径。**同一个实例上一次只运行一个 notebook**：
RTL 与 HLS 两份 notebook 会争用同一张 GPU 与同一个端口。

## 第 0 步 · 配置

这一格定义后面所有单元格共用的变量。一般只需要改 `MODEL`。

| 变量 | 含义 |
| --- | --- |
| `REPO` | 参考仓库克隆到哪里。`/workspace` 是持久盘，实例重启后仍在；其余目录重启后清空 |
| `REPO_URL` | 参考仓库的地址 |
| `OUT` | 本 notebook 的产物目录，按 track 分开，放在 `/tmp` |
| `MODEL` | 本地模型权重的**目录路径**（不是 Hugging Face 上的名字，评测环境断网）。放在持久盘 `/workspace` 下，重启后不用重新下载 |
| `VRAM_BUDGET_GB` | 推理服务预先占用的显存。比赛规定单卡不超过 32 GB，这里留出余量取 30 |
| `XILINXD_LICENSE_FILE` | EDA 工具的授权服务地址。镜像没有预设，这里补上，并写进 `/workspace/env.sh` 供终端使用 |
| `AGENT_TMP` 等 | EDA 工具的工作目录放到本地盘 `/var/tmp`。`/tmp` 是 4 GB 内存盘，连续运行容易写满 |

In [ ]:
import os, shutil

TRACK      = "rtl"
REPO_NAME  = "rtlagent2026"
REPO       = f"/workspace/{REPO_NAME}"
REPO_URL   = f"https://gitee.com/Vickyiii/{REPO_NAME}.git"
OUT        = f"/tmp/quickstart-{TRACK}"

MODEL          = "/workspace/models/Qwen3-8B"   # 持久盘，实例重启后仍在
VRAM_BUDGET_GB = "30"

os.environ.update(TRACK=TRACK, REPO=REPO, REPO_URL=REPO_URL, OUT=OUT,
                  AGENT_TMP="/var/tmp", SELFTEST_TMP="/var/tmp", EDA_TMP="/var/tmp/eda",
                  VRAM_BUDGET_GB=VRAM_BUDGET_GB)
os.environ.setdefault("XILINXD_LICENSE_FILE", "2100@vivado-lic.fpgachina.svc.cluster.local")
os.environ.setdefault("FLEXLM_TIMEOUT", "300000000")

# 授权变量只对本 notebook 有效。同时写进持久盘上的 /workspace/env.sh，
# 在终端里运行脚本前执行  source /workspace/env.sh  即可（见仓库 README 开头）
ENV_SH = "/workspace/env.sh"
if not os.path.exists(ENV_SH):
    with open(ENV_SH, "w") as fh:
        fh.write("export XILINXD_LICENSE_FILE=${XILINXD_LICENSE_FILE:-2100@vivado-lic.fpgachina.svc.cluster.local}\n"
                 "export FLEXLM_TIMEOUT=${FLEXLM_TIMEOUT:-300000000}\n")
if "AGENT_TMP" not in open(ENV_SH).read():      # 早先生成的 env.sh 没有这一行
    with open(ENV_SH, "a") as fh:
        fh.write("export EDA_TMP=/var/tmp/eda AGENT_TMP=/var/tmp SELFTEST_TMP=/var/tmp\n")

# 镜像通常已把 EDA 工具加进 PATH；若没有，按 XILINX_VIVADO 补上
if not all(shutil.which(t) for t in "xvlog xelab xsim vivado".split()):
    xroot = os.environ.get("XILINX_VIVADO")
    if xroot and os.path.isdir(f"{xroot}/bin"):
        os.environ["PATH"] = f"{xroot}/bin:" + os.environ["PATH"]

print("REPO =", REPO)
print("OUT  =", OUT)
print("XILINXD_LICENSE_FILE =", os.environ["XILINXD_LICENSE_FILE"])
print("终端用的环境文件:", ENV_SH)

## 第 1 步 · 检查实例环境

逐项列出实例上的资源，缺失的项目标为 `✗`。各部分怎么读：

| 部分 | 看什么 |
| --- | --- |
| `[GPU]` | 显卡型号与显存总量。验证阶段为 W7900 系列（48 GB），但方案须按单卡 32 GB 设计 |
| `[EDA 工具]` | 判定与智能体要用的命令都应为 `✓`，路径应在 `/tools/Xilinx/2026.1/` 下 |
| `[授权]` | 授权服务地址。地址存在不代表一定连得上，以第 6 步的判定结果为准 |
| `[推理栈]` | `torch` 与 `vllm` 的版本。vLLM 是第 9 步用来运行本地模型的推理服务 |
| `[存储]` | `/workspace` 是持久盘，放仓库与权重；`/var/tmp` 是本地盘，放 EDA 工作目录；`/tmp` 是 4 GB 内存盘，只放本 notebook 的小产物 |

In [ ]:
%%bash
ok(){ printf '  ✓ %-22s %s\n' "$1" "$2"; }
no(){ printf '  ✗ %-22s %s\n' "$1" "$2"; }

echo "[GPU]"
if command -v rocm-smi >/dev/null; then
    rocm-smi --showproductname --showmeminfo vram 2>/dev/null \
      | grep -E "Card Series|Card series|VRAM Total Memory" | sed 's/^/  /' | head -4
else no rocm-smi "未找到"; fi

echo "[EDA 工具]"
for t in xvlog xelab xsim vivado; do
    p=$(command -v $t) && ok "$t" "$p" || no "$t" "不在 PATH 中"
done
for v in XILINX_VIVADO XILINX_VITIS EDA_TMP; do
    [ -n "${!v:-}" ] && ok "$v" "${!v}" || no "$v" "未设置"
done

echo "[授权]"
[ -n "${XILINXD_LICENSE_FILE:-}" ] && ok XILINXD_LICENSE_FILE "$XILINXD_LICENSE_FILE" \
    || no XILINXD_LICENSE_FILE "未设置"

echo "[推理栈]"
python3 - <<'PY'
import importlib
for m in ("torch", "vllm"):
    try:
        mod = importlib.import_module(m)
        print(f"  ✓ {m:<22} {mod.__version__}")
    except Exception as e:
        print(f"  ✗ {m:<22} {type(e).__name__}")
PY

echo "[存储]"
df -h /workspace /var/tmp /tmp 2>/dev/null | sed 's/^/  /'

## 第 2 步 · 获取参考仓库

把参考仓库克隆到 `/workspace`；已经克隆过的，执行 `git pull` 拉取赛事方的最新修订。

输出怎么读：

- `git log` 那一行是当前所在的版本，向赛事方反馈问题时请附上
- 最后一行核对 `example/baseline.py` 的 SHA-256。基线脚本是增益项的参照，正式评测时
  会核对它是否为赛事方原版。显示 `✓` 说明你拿到的是当前发布版本；显示 `✗` 说明仓库
  不是最新，请重新执行本格

In [ ]:
%%bash
mkdir -p "$(dirname "$REPO")"
if [ -d "$REPO/.git" ]; then
    git -C "$REPO" pull --ff-only
else
    git clone "$REPO_URL" "$REPO"
fi
git -C "$REPO" log --oneline -1
# 部分下载方式会丢失可执行权限
find "$REPO" -name "*.sh" -exec chmod +x {} +

EXPECTED=17aaa609d4dbd36a94d1390e39f769394448f8588f3ccdd2bd37db9e34ed05e2
GOT=$(sha256sum "$REPO/example/baseline.py" | cut -d' ' -f1)
if [ "$GOT" = "$EXPECTED" ]; then
    echo "✓ baseline.py 为当前发布版本"
else
    echo "✗ baseline.py 与当前发布版本不一致（$GOT），请重新执行本格"
fi

## 第 3 步 · 仓库导览

参考仓库分四块。`example/` 是一套能完整跑通的示范方案，也是提交时的目录结构，可以直接
复制它作为起点。

| 路径 | 作用 | 需要改吗 |
| --- | --- | --- |
| `tasks/` | 3 道示例题，格式与正式评测题相同 | 不改 |
| `example/run.sh` | **方案入口**。读一道题，产出 `solution.v` 与 `trace.jsonl` | 一般不改 |
| `example/agent/` | **智能体**：`main.py` 控制流程（几轮、何时停）、`llm.py` 调用模型、`tools.py` 调用 EDA 工具、`prompts/` 提示词 | **主要改这里** |
| `example/skill/` | **技能包**：写给模型看的经验、检查清单与校验脚本 | **要改** |
| `example/run_baseline.sh`、`baseline.py` | **基线入口**，同一模型单次直接生成 | **不得修改** |
| `example/serve/vllm.sh` | 启动本地推理服务 vLLM 的参数 | 按所选模型改 |
| `example/serve/start.sh` | 把所有常驻服务（推理服务等）拉起来，就绪后返回 | 有自己的常驻服务时改 |
| `example/serve/serve_api.py` | **投题服务**：评测当天赛事方通过它向你投题 | 一般不改 |
| `example/model/MODEL.md` | 模型声明：来源、版本、量化方式、显存 | **必须填** |
| `example/REPORT.md` | 设计报告 | **必须写** |
| `selftest/run_selftest.sh` | 本地自测总入口：对每道题跑方案与基线，再判定、算分 | 不改 |
| `selftest/judge.py` | 判定器：给一份代码定级 | 不改 |
| `selftest/score.py` | 汇总各题级别，算出题集得分、能力分、增益分 | 不改 |
| `docs/` | 接口契约、常见问题、代码封存与提交说明 | 读 |
| `SCORING.md` | 评分细则 | 读 |

下一格列出仓库的实际内容，可以对照上表看。

In [ ]:
%%bash
cd "$REPO"
find . -maxdepth 3 -not -path './.git*' -not -path '*/__pycache__*' \
    | sort | sed 's|^\./||' | awk -F/ 'NR > 1 { printf "%*s%s\n", (NF-1)*2, "", $NF }'

## 第 4 步 · 安装示范方案的依赖

`example/requirements.txt` 列出示范方案用到的 Python 包：调用模型的 `requests`，以及
投题服务用的 `fastapi`、`uvicorn`、`pydantic`。输出最后一行显示「依赖安装完成」即可。

In [ ]:
%%bash
cd "$REPO/example"
grep -v '^#' requirements.txt | grep -v '^$'
pip install -q -r requirements.txt && echo "依赖安装完成"

## 第 5 步 · 认识一道题

每道题是 `tasks/` 下的一个目录。以 `ex01_popcount8` 为例：

| 文件 | 内容 | 智能体能看到吗 |
| --- | --- | --- |
| `prompt.txt` | 题面：要实现什么功能，模块名与端口清单也写在这里 | **能**，这是唯一的输入 |
| `task.json` | 题目元数据：顶层模块名、目标器件、时钟周期、各文件的名字 | 判定器用 |
| `top.txt` | 顶层模块名 | 判定器用 |
| `ref.sv` | 参考实现 | 不能，判定器用来比对 |
| `tb.sv` | 参考测试台，同时例化参考实现与你的模块，逐拍比对输出 | 不能 |
| `reference/solution.sv` | 一份正确答案，仅用于第 6 步验证判定链路 | 不能 |

**参考实现与测试台只有判定器能用，智能体拿不到。** 智能体若想在生成过程中验证自己的
代码，需要自己写测试，这正是本赛道考察的能力之一。

In [ ]:
%%bash
cd "$REPO/tasks/ex01_popcount8"
ls -l
echo
echo "=== prompt.txt（题面）"; cat prompt.txt
echo; echo "=== task.json（元数据）"; cat task.json

## 第 6 步 · 判定链路校验

这一步不涉及模型：把每道题自带的**正确答案**交给判定器，确认判定器在这台实例上能给出
满分。正确答案判不到满分，说明是环境（工具、授权）的问题，而不是代码的问题。

用到的脚本：`selftest/run_selftest.sh`。`--check` 只检查环境，`--reference` 对每道题的
参考答案调用一次判定器 `selftest/judge.py`，最后用 `selftest/score.py` 汇总。

**判定级别**（逐级递进，取达到的最高一级）：

| 级别 | 含义 | 判定方式 | 系数 |
| --- | --- | --- | --- |
| L0 | 未通过 | 交了空文件，或模块名、端口与题目不符，无法编译 | 0 |
| L1 | 可编译 | `xvlog` 分析、`xelab` 详细描述通过 | 0.2 |
| L2 | 仿真通过 | `xsim` 逐拍比对，输出与参考实现完全一致 | 0.7 |
| L3 | 可综合 | Vivado `synth_design` 在 ZU3EG 上综合通过 | 1.0 |

**输出的每一行**形如：

```
s0  ex01_popcount8       L3  CMS  coeff=1.0  28s  mismatches=0 samples=257
```

| 字段 | 含义 |
| --- | --- |
| `s0` | 第 0 次采样。正式评测每题采样 5 次，这里只有 1 次 |
| `L3` | 判定级别 |
| 字母组 | `CMS` 三个字母依次表示 Compile（编译）、Mismatch-free（仿真 0 失配）、Synth（综合）三个阶段，通过的显示字母，未通过或未走到的显示 `-` |
| `coeff=1.0` | 该级别对应的系数 |
| 秒数 | **判定**本身花的时间，不是解题时间 |
| `mismatches=0 samples=257` | 仿真共逐拍比对 257 拍，与参考实现不一致的有 0 拍 |

**期望结果**：3 道题全部判到 L3。

| 现象 | 原因 |
| --- | --- |
| 3 道均为 L3 | 判定链路正常，可以继续 |
| 出现 `LICENSE_ERROR` | 授权签不出来。`xsim` 与 `synth_design` 需要签出授权，`xvlog`、`xelab` 不需要，所以授权不通时，典型现象是全部卡在 L1。属环境问题，请在赛道 QQ 群反馈 |
| 出现 `PART_NOT_LICENSED` | 授权档位不覆盖目标器件，同属环境问题 |
| 其他低于 L3 的结果 | 查看 `selftest/out/logs/` 下的工具日志 |

In [ ]:
%%bash
cd "$REPO/selftest"
./run_selftest.sh --check && ./run_selftest.sh --reference

## 第 7 步 · 桩模式：验证 `run.sh` 的输入输出

`run.sh` 是方案的唯一入口，约定很简单：

```
run.sh <题目目录> <输出目录>
   读  <题目目录>/prompt.txt（HLS 另有 interface.txt）
   写  <输出目录>/solution.v      产出的代码，做不出来就写空文件
       <输出目录>/trace.jsonl     每一次模型调用与工具调用的记录
```

`LLM_BACKEND=mock` 让智能体使用一个「假模型」：不联网、不需要 GPU，直接返回一段格式
正确的空壳代码。这一步只验证「读题 → 写产物」这条链路是通的，**不验证解题能力**，所以
判定结果低于 L3 是正常的。

`trace.jsonl` 每行是一条 JSON 记录，下一格会把它逐行列出。常见字段：

| 字段 | 含义 |
| --- | --- |
| `tool` | 这一步做了什么：`llm` 调用模型，`extract` 从回复中抽取代码，以及工具 `check_interface`（核对模块名与端口，不调用 EDA 工具）、`lint`（用 `xvlog` / `xelab` 编译检查）、`synth`（用 Vivado 综合） |
| `round` | 第几轮。智能体每轮让模型写一次代码，检查不通过就把报错交给模型修正 |
| `rc` | 工具的返回码，0 表示通过 |
| `excerpt` | 工具输出的摘要，通常是报错信息 |
| `tokens_in` / `tokens_out` | 模型这一次读了多少 token、写了多少 token |

trace 是工程质量评定的依据之一，也是你排查「智能体为什么没做出来」的第一手材料。

In [ ]:
%%bash
cd "$REPO/example"
rm -rf "$OUT/mock" && mkdir -p "$OUT/mock"
LLM_BACKEND=mock ./run.sh ../tasks/ex01_popcount8 "$OUT/mock"
echo "=== 输出目录"; ls -l "$OUT/mock"
echo "=== solution.v（前 20 行）"; head -20 "$OUT/mock/solution.v"
echo "=== trace.jsonl（逐行）"
python3 - "$OUT/mock/trace.jsonl" <<'PY'
import json, sys
for line in open(sys.argv[1]):
    e = json.loads(line)
    keys = [k for k in ("tool", "event", "round", "rc", "tokens_in", "tokens_out") if k in e]
    head = "  ".join(f"{k}={e[k]}" for k in keys)
    ex = (e.get("excerpt") or "").replace("\n", " ")[:80]
    print(head + (f"  | {ex}" if ex else ""))
PY
echo "=== 判定"
cd ../selftest && python3 judge.py --task ../tasks/ex01_popcount8 --solution "$OUT/mock/solution.v"

## 第 8 步 · 投题服务：评测当天怎么收题

正式评测时，赛事方不会登录你的实例，而是通过 HTTP 把题目一道一道发过来。接收题目的
是 `example/serve/serve_api.py`，它守在 `127.0.0.1:7860`，每收到一道题就在后台调用一次
`run.sh`，做完后等赛事方来取结果。

| 接口 | 作用 |
| --- | --- |
| `GET /v1/health` | 赛事方投题前确认服务就绪，返回 `ready`、`track`、`model`、`vram_gb` |
| `POST /v1/solve` | 投一道题。请求里有 `task_id`、`prompt`、`interface`、`deadline_s` 等，**立即**返回一个任务编号 `job_id` |
| `GET /v1/jobs/<job_id>` | 查询这道题。`status` 为 `queued`（排队）、`running`（解题中）或 `done`（完成）；`done` 时附带 `solution`、`trace`、`elapsed_s` |

投题之所以是「先拿编号、再查询」，是因为一道题常要几分钟，而平台入口会切断 60 秒内
没有响应的请求。赛事方每 2 秒查询一次。

两条需要记住的规则：

- 每个请求都要带 `Authorization: Bearer <令牌>`。令牌由赛事方发放，用环境变量
  `FPGACHINA_TOKEN` 交给服务
- `deadline_s` 是单题时限，从投题时算起。服务会在到点前结束 `run.sh` 并交出已有的结果；
  投题后超过 `deadline_s + 20` 秒仍未查询到 `done`，该题按 L0 计

下一格用桩模型在本机启动投题服务，调用一次 `/v1/health`，投一道题并每 2 秒查询一次，
最后关掉服务。输出里能看到状态从 `running` 变为 `done` 的过程。
`vram_gb` 在桩模式下为 0，因为还没有加载模型。完整的接口定义见 `docs/API_CONTRACT.md`，
暴露到公网的方法见 `example/serve/README.md`。

In [ ]:
import subprocess, sys, time, json, signal, socket, urllib.request

# 7860 已被占用（例如之前在终端里手动启动过投题服务）时，本格起不来自己的服务
with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", 7860)) == 0:
        raise RuntimeError("7860 端口已被占用，多半是之前手动启动的投题服务。"
                           "先在终端执行  pkill -f 'uvicorn serve.serve_api'  再重新运行本格")

TOKEN = "quickstart-local-test"
HDR   = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

def call(path, body=None, timeout=10):
    req = urllib.request.Request("http://127.0.0.1:7860" + path, headers=HDR,
                                 data=None if body is None else json.dumps(body).encode())
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.load(r)

task   = f"{REPO}/tasks/ex01_popcount8"
prompt = open(f"{task}/prompt.txt").read()
iface  = open(f"{task}/interface.txt").read() if os.path.exists(f"{task}/interface.txt") else ""
log    = f"{OUT}-serve_api.log"

api = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "serve.serve_api:app", "--host", "127.0.0.1", "--port", "7860"],
    cwd=f"{REPO}/example", stdout=open(log, "w"), stderr=subprocess.STDOUT,
    start_new_session=True,
    env=dict(os.environ, FPGACHINA_TOKEN=TOKEN, MODEL_NAME="mock", LLM_BACKEND="mock"))
try:
    for _ in range(30):
        try:
            print("GET /v1/health ->", call("/v1/health"))
            break
        except Exception:
            if api.poll() is not None:
                raise RuntimeError(f"投题服务已退出，见 {log}")
            time.sleep(1)

    t0 = time.time()
    job = call("/v1/solve", dict(task_id="demo-1", nonce="n1", mode="agent", prompt=prompt,
                                 interface=iface, deadline_s=900))
    print(f"\nPOST /v1/solve -> {job}")

    last = None
    while True:
        time.sleep(2)
        r = call(f"/v1/jobs/{job['job_id']}")
        if r["status"] != last:
            print(f"  {time.time() - t0:5.1f} 秒  GET /v1/jobs/... -> status = {r['status']}")
            last = r["status"]
        if r["status"] == "done":
            break

    print("\n结果：")
    print("  task_id  :", r["task_id"])
    print("  solution :", len(r["solution"]), "字节，前 3 行：")
    for line in r["solution"].splitlines()[:3]:
        print("             ", line)
    print("  trace    :", len(r["trace"].splitlines()), "条记录")
    print("  elapsed_s:", r["elapsed_s"], "（服务自报，仅供核对；计分以赛事方测得的时间为准）")
finally:
    if api.poll() is None:
        os.killpg(api.pid, signal.SIGTERM)
        api.wait(timeout=30)
    print("\n投题服务已关闭")

## 第 9 步 · 本地模型：一道题的方案与基线

正式评测在断网环境中进行，必须使用**本地部署的开源权重**。本步骤分三格：

1. （可选）下载权重
2. 用 `example/serve/start.sh` 启动推理服务
3. 对同一道题分别运行方案 `run.sh` 与基线 `run_baseline.sh`，并判定

**`start.sh` 做什么**：调用 `serve/vllm.sh` 在后台启动 vLLM，等它加载完权重、能够应答后
才返回；服务已在运行时直接返回。它还会启动一个定期清理 `/var/tmp/eda` 临时目录的后台任务。
vLLM 的日志在 `/var/tmp/serve-logs/vllm.log`，启动失败时先看这里。

**`vllm.sh` 的几个关键参数**：`--model` 是权重目录；`--max-model-len` 是上下文长度；
`--gpu-memory-utilization` 由 `VRAM_BUDGET_GB` 换算，vLLM 会按它**一次性预占**显存，与
模型大小无关。这些配置须如实写进 `MODEL.md`，基线与方案必须使用同一个推理服务。

**思考设置与输出上限**：Qwen3 这类推理模型可以先输出一段思考过程再给出答案，思考本身就
可能有数千 token；上限设小了，模型还没写到代码就被截断，智能体只能交空文件，而且不会报错。
是否开启思考在 `example/manifest.json` 的 `generation` 段声明（示范为关闭，
`"enable_thinking": false`），基线与智能体都按这一声明发请求，两者必须一致。输出上限设为
16384，与基线相同。

下面这格下载权重，约 16 GB，存到持久盘 `/workspace/models`。权重已在 `MODEL` 目录下的，跳过这一格。

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen3-8B", local_dir=MODEL)

In [ ]:
import subprocess, urllib.request, json

assert os.path.isdir(MODEL), f"找不到权重目录 {MODEL}，请先下载或修改第 0 步的 MODEL"
r = subprocess.run(["./serve/start.sh"], cwd=f"{REPO}/example",
                   env=dict(os.environ, MODEL=MODEL, START_TIMEOUT_S="1200"))
if r.returncode != 0:
    raise RuntimeError("start.sh 失败，见 /var/tmp/serve-logs/vllm.log")
with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5) as resp:
    served = json.load(resp)["data"][0]["id"]
print("vLLM 就绪，模型名：", served)

# 智能体与基线都通过这几个变量找到推理服务
os.environ.update(LLM_BACKEND="openai", LLM_BASE_URL="http://127.0.0.1:8000/v1",
                  LLM_API_KEY="dummy", LLM_MODEL=served, MODEL_NAME=served,
                  LLM_MAX_TOKENS="16384")

下一格对 `ex01_popcount8` 分别运行方案与基线，再判定两份产物。输出怎么读：

| 输出 | 含义 |
| --- | --- |
| `real` | 这一次运行的墙钟时间。方案通常要多轮调用模型与工具，比基线慢 |
| 显存实测 | 推理服务加载后的显存占用，须在 32 GB 以内，填进 `MODEL.md` |
| 判定两行 | 第一行是方案，第二行是基线，格式同第 6 步 |
| 模型调用记录 | 每次调用的 `tokens_out` 与 `finish`。`finish=length` 表示撞上了输出上限被截断 |

一道题、一次采样的结果有偶然性，方案某次不如基线也不奇怪，这一步只确认整条链路能跑通。

In [ ]:
%%bash
cd "$REPO/example"
T=../tasks/ex01_popcount8
rm -rf "$OUT/agent" "$OUT/base"
echo "=== 方案 run.sh";          time ./run.sh          $T "$OUT/agent"
echo "=== 基线 run_baseline.sh"; time ./run_baseline.sh $T "$OUT/base"
echo "=== 显存实测"; rocm-smi --showmemuse --showmeminfo vram 2>/dev/null | grep -i "used" | head -2
echo "=== 判定（第一行方案，第二行基线）"
cd ../selftest
python3 judge.py --task $T --solution "$OUT/agent/solution.v"
python3 judge.py --task $T --solution "$OUT/base/solution.v"
echo "=== 模型调用记录"
for who in agent base; do
    python3 - "$OUT/$who/trace.jsonl" "$who" <<'PY'
import json, sys
for line in open(sys.argv[1]):
    e = json.loads(line)
    if e.get("tool") == "llm":
        print(f"  {sys.argv[2]:5s} round={e.get('round')}  tokens_in={e.get('tokens_in')}  "
              f"tokens_out={e.get('tokens_out')}  finish={e.get('finish', '-')}")
PY
done

## 第 10 步 · 完整自测与算分（可选，约 20 分钟）

对 3 道示例题都跑一遍方案与基线，各自判定，再用 `selftest/score.py` 算分。每个样本都要
真实调用模型与 EDA 工具，全部串行，所以耗时较长；期间输出会停顿，属正常现象。第 9 步
已经证明链路可用，这一步可以留到改完智能体之后再跑。

输出分三段：

1. **逐题结果**：`agent`（方案）、`baseline`（基线）各一段，每行格式同第 6 步
2. **汇总表**：每道题的级别与 `pass@1`。`pass@1` 是该题各次采样系数的平均值，是计分口径；
   `pass@5` 取最好一次，只作稳定性参考，采样 1 次时二者相等
3. **算分**，例如：

```
能力  = 30 × 0.400 = 12.0 / 30
增益  g = (0.400 − 0.200) / (1 − 0.200) = 0.250
       40 × min(0.250 / 0.6, 1) = 16.7 / 40
       诊断：倍数 0.400 / 0.200 = 2.00，不计分
```

读法：方案题集得分 0.400，能力分 12.0；基线 0.200，与满分 1.0 之间差 0.800，方案补上了
其中的 0.200，即 g = 0.25；补上六成（g = 0.6）即得增益满分 40 分，这里得 16.7 分。代价与
工程质量两项本地无法自评。

**3 道题、每题 1 次采样，任何一道题的结果变化都会让分数大幅波动**，只能用来确认算分流程
可用。要得到可比的数字，请用更多题目（见 `selftest/README.md`）并把 `--samples 1` 改为
`--samples 5`。

In [ ]:
%%bash
cd "$REPO/selftest" && ./run_selftest.sh --samples 1

## 第 11 步 · 清理

停止 vLLM，释放显存。vLLM 由 `start.sh` 在后台启动，不是本 notebook 的子进程，所以按进程名
结束。要继续用模型的话，跳过这一格。

In [ ]:
%%bash
if pkill -f vllm.entrypoints.openai.api_server; then
    sleep 5; echo "vLLM 已停止"
else
    echo "vLLM 未在运行"
fi

## 接下来

| 想做的事 | 看哪里 |
| --- | --- |
| 改进智能体与技能包 | `example/README.md`；`example/skill/README.md` |
| 理解评测当天的投题方式 | `docs/API_CONTRACT.md` |
| 代码封存与仓库提交（Gitee 或 GitHub） | `docs/SUBMISSION.md` |
| 评分细则 | `SCORING.md` |
| 常见问题 | `docs/FAQ.md` |

### 常见问题速查

| 现象 | 先查什么 |
| --- | --- |
| 运行 `.sh` 脚本报 `Permission denied` | 重新执行第 2 步，它会补上可执行权限 |
| 第 6 步出现 `LICENSE_ERROR` 或全部卡在 L1 | 授权服务连不上，在 QQ 群反馈，附第 1 步与第 6 步的输出 |
| `start.sh 失败` | `tail -30 /var/tmp/serve-logs/vllm.log` |
| 方案交了空文件（判定 L0、耗时 0 秒） | 看 trace 里 `llm` 记录的 `tokens_out` 是否等于输出上限，是则被截断，调大 `LLM_MAX_TOKENS` |
| 投题服务起不来 | `/tmp/quickstart-rtl-serve_api.log` 里的报错；7860 端口是否已被占用 |
| 实例重启后接着用 | 重启会停掉所有服务，并清空 `/workspace` 以外的内容（包括 pip 安装的包）。仓库与权重都在 `/workspace`，从第 0 步重新执行即可，第 2 步只是拉取更新，第 9 步跳过下载 |

环境问题请在赛道 QQ 群反馈，并附上第 1、2、6 步的完整输出。